# ДЗ №7
### *виконала студентка групи ФБ-33 Журавльова Марія* ###

Нагадаю, що мій набір даних - TMDB 10000 Movies Dataset (https://www.kaggle.com/datasets/i0xc0d3x00000/tmdb-10000-movies-dataset)

**Опис колонок:**

- title - назва фільму;
- overview - опис сюжету фільму;
- release_date - оригінальна дата випуску;
- vote_average - середній рейтинг фільму;
- vote_count - кількість отриманих голосів;
- original_language - мова зйомки;
- popularity - індекс популярності.

- **Розробіть етапи АБ-тестування для вашого прикладу.**
- **Задайте небхідні параметри і метрики прописані в етапах.**

1. *Сформулювати гіпотези.*

Гіпотеза H0: середня популярність групи B не відрізняється від групи A.
Гіпотеза H1: середня популярнвсть групи В виза за групу А.

2. *Визначення метрики.*

Метрика: середнє значення popularity для кожної групи.

3. *Затвердити критерії успіху та дії.*

Критерій успіху:
- статистична значущість: P-value має бути меншим за рівень значущості 0.05
- практична значущість: середнє значення популярності групи В має бути вищим за групу А, наприклад, на 15%.

4. *Підготувати експеримент.*

Вибірка А: рандомні фільми.
Вибірка В: фільми з високими vote_average.
Мінімальний розмір вибірки: фактичний розмір N, який визначаэться розміром меншої групи (у моєму випадку він склав N=3727).

5. *Запустити експеримент.*

Формування вибірок: фактичний розмір групи В, що відповідає критерію vote_average >=7, виявився 3727 фільмів. Для збалансованого формування вибірок, група А формується шляхом випадкового вибору тієї ж самої кількості фільмів з усього датасету.

6. *Проаналізувати результати та зробити висновки.*

- обчислити середні занчення для popularity для А та В.
- отримати p-value
- прийняти або відхилити H0, спираючись на p-value та практичну значущість.

- **Згенеруйте 2 вибірки (рандомна та з наперед заданими параметрами).**


In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\User\OneDrive\Desktop\university\МТАД\movies-tmdb-10000.csv")
df = df.dropna(subset=['popularity', 'vote_average']).reset_index(drop=True)

np.random.seed(42)

high_rating = df[df['vote_average'] >= 7]
n_b = len(high_rating)
sample_B = high_rating.sample(n=n_b, replace=False, random_state=42)

n_a = n_b
sample_A = df.sample(n=n_a, replace=False, random_state=42)

print(f"Розмір вибірки А: {len(sample_A)}")
print(f"Розмір вибірки B: {len(sample_B)}")

print(f"Середня популярність в А: {sample_A['popularity'].mean():.4f}")
print(f"Середня популярність в В: {sample_B['popularity'].mean():.4f}")

Розмір вибірки А: 3727
Розмір вибірки B: 3727
Середня популярність в А: 23.1379
Середня популярність в В: 29.2770


- **Порівняйте на виході метрики та виконання критерію успіху.**


***Практична значущість:***

In [12]:
mean_a = sample_A['popularity'].mean()
mean_b = sample_B['popularity'].mean()

lift = (mean_b / mean_a - 1) * 100
print(f"Приріст популярності: {lift:.2f}%")

Приріст популярності: 26.53%


Отже, практична значущість виконана, оскільки 26.53% > 15%

***Статистична значущість:***

In [17]:
from scipy.stats import ttest_ind

A = sample_A['popularity']
B = sample_B['popularity']

t_stat, p_value = ttest_ind(B, A, equal_var=False)
print(f"t-stat = {t_stat:.4f}, p-value = {p_value:.4f}")

t-stat = 3.7246, p-value = 0.0002


Оскільки ttest_ind за замовчуванням видає двостороннє p-value, нам треба додатково поділити на 2. Це можливо лише тоді, коли середне значення у групі B > середнього значення у групі А (дійсно 29.2770 > 23.1379). Отже, отримую p-value = 0.0001 що << 0.05 

**Висновок:** оскільки стратегія підбору фільмів призвела до статистично значущого та практично значущого підвищення середньої популярності на 26.53%, то це дозволяє нам відхилити гіпотезу H0 на користь H1.